# CSIRO Biomass - v3 Fixed Inference (T4×2)

## 🔧 修正版推論コード

### 修正内容
1. **KaggleCompatibleMambaBlock**: 外部依存なしの実装
2. **統一ディメンション**: POOL_OUTPUT_DIM = 1920
3. **物理制約**: 訓練と推論で統一
4. **安全なモデルロード**: エラーハンドリング強化

In [ ]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'max_split_size_mb:128'
os.environ['CUDA_LAUNCH_BLOCKING'] = '0'

import gc
import time
import random
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import timm
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# GPU最適化
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# Seed設定
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# GPU確認
n_gpus = torch.cuda.device_count()
print(f"Available GPUs: {n_gpus}")
for i in range(n_gpus):
    props = torch.cuda.get_device_properties(i)
    free_mem = torch.cuda.mem_get_info(i)[0] / 1024**3
    total_mem = props.total_memory / 1024**3
    print(f"GPU {i}: {props.name} ({free_mem:.1f}/{total_mem:.1f} GB free)")

device0 = torch.device("cuda:0")
device1 = torch.device("cuda:1" if n_gpus > 1 else "cuda:0")

In [ ]:
class CFG:
    TARGETS = ["Dry_Green_g", "Dry_Dead_g", "Dry_Clover_g", "GDM_g", "Dry_Total_g"]
    DATA_DIR = Path("/kaggle/input/csiro-biomass")
    MODEL_DIR = Path("/kaggle/input/csiro-v3-complete-fix")
    
    # モデル設定（訓練と統一）
    IMG_SIZE = 384
    N_FOLDS = 5
    BACKBONE = "vit_huge_plus_patch16_dinov3.lvd1689m"
    BACKBONE_DIM = 1280
    POOL_OUTPUT_DIM = int(BACKBONE_DIM * 1.5)  # 1920 - 訓練と統一
    BATCH_SIZE = 2
    NUM_WORKERS = 4
    
    # GPU割り当て
    GPU0_FOLDS = [0, 2, 4]
    GPU1_FOLDS = [1, 3]
    
    # TTA設定
    USE_TTA = True
    TTA_TRANSFORMS = ["original", "hflip", "vflip"]
    TTA_WEIGHTS = [1.0, 0.7, 0.7]
    
    # 推論最適化
    USE_FP16 = True
    USE_CHANNELS_LAST = True
    
    # アンサンブル重み（訓練と統一）
    FOLD_WEIGHTS = [1.0, 0.9, 1.0, 1.1, 0.95]

## Fixed Model Architecture

In [ ]:
# Kaggle互換Mambaブロック（外部依存なし）
class KaggleCompatibleMambaBlock(nn.Module):
    """Kaggle環境で動作するMamba風ブロック（GRU使用）"""
    def __init__(self, dim, d_state=16, expand=2, dt_rank="auto", conv_size=4):
        super().__init__()
        self.dim = dim
        d_inner = int(dim * expand)
        dt_rank = d_inner // 16 if dt_rank == "auto" else dt_rank
        
        # 入力投影
        self.in_proj = nn.Linear(dim, d_inner * 2)
        
        # 1D畳み込み（因果的）
        self.conv1d = nn.Conv1d(
            in_channels=d_inner,
            out_channels=d_inner,
            kernel_size=conv_size,
            padding=conv_size - 1,
            groups=d_inner
        )
        
        # GRUで状態空間モデルを近似
        self.gru = nn.GRU(
            input_size=d_inner,
            hidden_size=d_state,
            num_layers=1,
            batch_first=True,
            bidirectional=False
        )
        
        # 状態から特徴への投影
        self.state_to_feat = nn.Linear(d_state, d_inner)
        
        # 出力投影
        self.out_proj = nn.Linear(d_inner, dim)
        self.dropout = nn.Dropout(0.1)
        
    def forward(self, x):
        # x: (B, L, D)
        B, L, D = x.shape
        
        # ゲート付き投影
        x_proj = self.in_proj(x)
        x_inner, gate = x_proj.chunk(2, dim=-1)
        x_inner = F.silu(x_inner)
        gate = torch.sigmoid(gate)
        
        # 1D畳み込み
        x_conv = x_inner.transpose(1, 2)
        x_conv = self.conv1d(x_conv)[:, :, :L]
        x_conv = x_conv.transpose(1, 2)
        
        # GRUで処理
        h_state, _ = self.gru(x_conv)
        h_feat = self.state_to_feat(h_state)
        
        # ゲート適用
        y = h_feat * gate
        
        # 出力投影
        output = self.out_proj(y)
        output = self.dropout(output)
        
        return output + x  # 残差接続


class SpatialAwarePooling(nn.Module):
    """空間情報を保持するプーリング"""
    def __init__(self, dim=1280, output_dim=1920):
        super().__init__()
        self.dim = dim
        self.output_dim = output_dim
        
        # アテンション
        self.attention = nn.Sequential(
            nn.Linear(dim, dim // 4),
            nn.GELU(),
            nn.Linear(dim // 4, 1)
        )
        
        # 空間投影（output_dimに合わせる）
        spatial_dim = output_dim - dim
        self.spatial_proj = nn.Linear(dim, spatial_dim // 2)
        
    def forward(self, x):
        # x: (B, L, D)
        # アテンション重み
        attn_weights = F.softmax(self.attention(x), dim=1)
        weighted_mean = torch.sum(x * attn_weights, dim=1)
        
        # 空間特徴
        spatial_feat = self.spatial_proj(x)
        spatial_max = torch.max(spatial_feat, dim=1)[0]
        spatial_avg = torch.mean(spatial_feat, dim=1)
        
        # 結合（output_dim = 1920）
        return torch.cat([weighted_mean, spatial_max, spatial_avg], dim=1)


class CrossAttentionStereoFusion(nn.Module):
    """左右画像の相互作用"""
    def __init__(self, dim=1280):
        super().__init__()
        self.scale = dim ** -0.5
        
        # 軽量クロスアテンション
        self.q_proj = nn.Linear(dim, dim // 4)
        self.k_proj = nn.Linear(dim, dim // 4)
        self.v_proj = nn.Linear(dim, dim // 4)
        self.out_proj = nn.Linear(dim // 4, dim)
        self.dropout = nn.Dropout(0.1)
        
    def forward(self, left_feat, right_feat):
        # left_feat, right_feat: (B, L, D)
        B, L, D = left_feat.shape
        
        # プール
        left_pool = left_feat.mean(1, keepdim=True)
        right_pool = right_feat.mean(1, keepdim=True)
        
        # クロスアテンション（左→右）
        q = self.q_proj(left_feat)
        k = self.k_proj(right_pool)
        v = self.v_proj(right_pool)
        
        attn = torch.matmul(q, k.transpose(-2, -1)) * self.scale
        attn = F.softmax(attn, dim=-1)
        left_enhanced = self.out_proj(torch.matmul(attn, v))
        left_enhanced = left_feat + self.dropout(left_enhanced)
        
        # クロスアテンション（右→左）
        q = self.q_proj(right_feat)
        k = self.k_proj(left_pool)
        v = self.v_proj(left_pool)
        
        attn = torch.matmul(q, k.transpose(-2, -1)) * self.scale
        attn = F.softmax(attn, dim=-1)
        right_enhanced = self.out_proj(torch.matmul(attn, v))
        right_enhanced = right_feat + self.dropout(right_enhanced)
        
        return torch.cat([left_enhanced, right_enhanced], dim=1)


class FixedInferenceModel(nn.Module):
    """修正版推論モデル"""
    def __init__(self, model_name, pretrained=False):
        super().__init__()
        self.backbone = timm.create_model(
            model_name, pretrained=pretrained,
            num_classes=0, global_pool=""
        )
        
        nf = self.backbone.num_features  # 1280
        pool_dim = CFG.POOL_OUTPUT_DIM  # 1920
        
        # 修正版モジュール
        self.stereo_fusion = CrossAttentionStereoFusion(nf)
        self.mamba_fusion = nn.Sequential(
            KaggleCompatibleMambaBlock(nf * 2),
            KaggleCompatibleMambaBlock(nf * 2)
        )
        self.spatial_pool = SpatialAwarePooling(nf * 2, pool_dim)
        
        # ヘッド
        self.head = nn.Sequential(
            nn.Linear(pool_dim, pool_dim // 2),
            nn.LayerNorm(pool_dim // 2),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(pool_dim // 2, pool_dim // 4),
            nn.LayerNorm(pool_dim // 4),
            nn.GELU(),
            nn.Linear(pool_dim // 4, 5),
            nn.Softplus()
        )
        
    def forward(self, x):
        left, right = x
        
        # Channels last
        if CFG.USE_CHANNELS_LAST:
            left = left.contiguous(memory_format=torch.channels_last)
            right = right.contiguous(memory_format=torch.channels_last)
        
        # Backbone
        x_l = self.backbone(left)
        x_r = self.backbone(right)
        
        # Fusion
        x = self.stereo_fusion(x_l, x_r)
        x = self.mamba_fusion(x)
        x = self.spatial_pool(x)
        
        # Head
        out = self.head(x)
        
        # 物理制約
        green, dead, clover = out[:, 0:1], out[:, 1:2], out[:, 2:3]
        gdm = green + clover
        total = green + dead + clover
        
        return torch.cat([green, dead, clover, gdm, total], dim=1)

## Dataset

In [ ]:
class TestDataset(Dataset):
    """テストデータセット"""
    def __init__(self, df, data_dir, img_size=384, tta_type="original"):
        self.df = df.reset_index(drop=True)
        self.data_dir = Path(data_dir)
        self.img_size = img_size
        self.tta_type = tta_type
        
        # 正規化パラメータ
        self.mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
        self.std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = self.data_dir / row["image_path"]
        
        # PIL読み込み
        with Image.open(img_path) as img:
            img = img.convert("RGB")
            
            # TTA
            if self.tta_type == "hflip":
                img = img.transpose(Image.FLIP_LEFT_RIGHT)
            elif self.tta_type == "vflip":
                img = img.transpose(Image.FLIP_TOP_BOTTOM)
            
            # リサイズ
            img = img.resize((self.img_size * 2, self.img_size), Image.BILINEAR)
            
            # Split
            w = img.width
            left = img.crop((0, 0, w // 2, self.img_size))
            right = img.crop((w // 2, 0, w, self.img_size))
        
        # Tensor変換
        left = T.ToTensor()(left)
        right = T.ToTensor()(right)
        
        # 正規化
        left = (left - self.mean) / self.std
        right = (right - self.mean) / self.std
        
        return left, right, row["image_path"]


def collate_fn(batch):
    """バッチ処理"""
    lefts = torch.stack([b[0] for b in batch])
    rights = torch.stack([b[1] for b in batch])
    paths = [b[2] for b in batch]
    return lefts, rights, paths

## Safe Model Loading

In [ ]:
def safe_load_model(model_path, device, model):
    """安全なモデルロード"""
    try:
        # 重みロード
        state_dict = torch.load(model_path, map_location=device, weights_only=True)
        # save_model() のラップ形式チェックポイントに対応
        if isinstance(state_dict, dict) and "model_state_dict" in state_dict:
            state_dict = state_dict["model_state_dict"]
        
        # module.プレフィックス除去
        if list(state_dict.keys())[0].startswith("module."):
            state_dict = {k.replace("module.", ""): v for k, v in state_dict.items()}
        
        # 互換性チェック
        model_state = model.state_dict()
        filtered_state = {}
        missing_keys = []
        unexpected_keys = []
        
        for k, v in state_dict.items():
            if k in model_state:
                if v.shape == model_state[k].shape:
                    filtered_state[k] = v
                else:
                    print(f"  Shape mismatch: {k} - checkpoint: {v.shape}, model: {model_state[k].shape}")
            else:
                unexpected_keys.append(k)
        
        for k in model_state.keys():
            if k not in filtered_state:
                missing_keys.append(k)
        
        # ロード
        model.load_state_dict(filtered_state, strict=False)
        
        if missing_keys:
            print(f"  Missing keys: {len(missing_keys)}")
        if unexpected_keys:
            print(f"  Unexpected keys: {len(unexpected_keys)}")
        
        return True
        
    except Exception as e:
        print(f"  Error loading model: {e}")
        return False

## Inference Function

In [ ]:
@torch.inference_mode()
def inference_fold(fold, device, test_wide):
    """Fold毎の推論"""
    
    # モデルパス確認
    model_path = CFG.MODEL_DIR / f"best_ema_fold{fold}.pth"
    if not model_path.exists():
        model_path = CFG.MODEL_DIR / f"best_fold{fold}.pth"
    
    if not model_path.exists():
        print(f"Fold {fold}: model not found")
        return None
    
    print(f"\nFold {fold}: Loading model...")
    start_time = time.time()
    
    # メモリクリア
    torch.cuda.empty_cache()
    
    # モデル作成
    model = FixedInferenceModel(CFG.BACKBONE, pretrained=False)
    
    # 安全にロード
    if not safe_load_model(model_path, device, model):
        print(f"  Failed to load model for fold {fold}")
        return None
    
    model = model.to(device)
    model.eval()
    
    # FP16変換
    if CFG.USE_FP16:
        model = model.half()
    
    # Channels last
    if CFG.USE_CHANNELS_LAST:
        model = model.to(memory_format=torch.channels_last)
    
    print(f"  Model loaded in {time.time() - start_time:.1f}s")
    
    all_tta_preds = []
    
    # TTA推論
    for tta_idx, tta_type in enumerate(CFG.TTA_TRANSFORMS if CFG.USE_TTA else ["original"]):
        print(f"  TTA {tta_type}...", end=" ")
        tta_start = time.time()
        
        # データセット
        dataset = TestDataset(test_wide, CFG.DATA_DIR, CFG.IMG_SIZE, tta_type)
        loader = DataLoader(
            dataset,
            batch_size=CFG.BATCH_SIZE,
            shuffle=False,
            num_workers=CFG.NUM_WORKERS,
            collate_fn=collate_fn,
            pin_memory=True,
            prefetch_factor=2
        )
        
        preds = []
        
        # バッチ推論
        for batch_idx, (left, right, _) in enumerate(loader):
            # GPU転送
            left = left.to(device, non_blocking=True)
            right = right.to(device, non_blocking=True)
            
            if CFG.USE_FP16:
                left = left.half()
                right = right.half()
            
            if CFG.USE_CHANNELS_LAST:
                left = left.contiguous(memory_format=torch.channels_last)
                right = right.contiguous(memory_format=torch.channels_last)
            
            # 推論
            out = model((left, right))
            preds.append(out.float().cpu().numpy())
            
            # メモリクリア
            if batch_idx % 50 == 0:
                torch.cuda.empty_cache()
        
        preds = np.vstack(preds)
        all_tta_preds.append(preds)
        
        print(f"Done in {time.time() - tta_start:.1f}s")
    
    # TTA平均
    if CFG.USE_TTA:
        weighted_preds = []
        for pred, weight in zip(all_tta_preds, CFG.TTA_WEIGHTS):
            weighted_preds.append(pred * weight)
        final_preds = np.sum(weighted_preds, axis=0) / sum(CFG.TTA_WEIGHTS)
    else:
        final_preds = all_tta_preds[0]
    
    # クリーンアップ
    del model
    torch.cuda.empty_cache()
    gc.collect()
    
    print(f"  Fold {fold} total time: {time.time() - start_time:.1f}s")
    
    return final_preds

## Main Inference

In [ ]:
# テストデータ読み込み
test_df = pd.read_csv(CFG.DATA_DIR / "test.csv")
print(f"Test samples: {len(test_df)}")
test_wide = test_df[["image_path"]].drop_duplicates().reset_index(drop=True)
print(f"Unique test images: {len(test_wide)}")

# 推論開始
total_start = time.time()
all_preds = []

# GPU0処理
print(f"\n{'='*50}")
print(f"GPU 0: Processing folds {CFG.GPU0_FOLDS}")
print(f"{'='*50}")

for fold in CFG.GPU0_FOLDS:
    preds = inference_fold(fold, device0, test_wide)
    if preds is not None:
        all_preds.append((fold, preds * CFG.FOLD_WEIGHTS[fold]))

# GPU1処理
if n_gpus > 1:
    print(f"\n{'='*50}")
    print(f"GPU 1: Processing folds {CFG.GPU1_FOLDS}")
    print(f"{'='*50}")
    
    for fold in CFG.GPU1_FOLDS:
        preds = inference_fold(fold, device1, test_wide)
        if preds is not None:
            all_preds.append((fold, preds * CFG.FOLD_WEIGHTS[fold]))
else:
    print(f"\nSingle GPU mode...")
    for fold in CFG.GPU1_FOLDS:
        preds = inference_fold(fold, device0, test_wide)
        if preds is not None:
            all_preds.append((fold, preds * CFG.FOLD_WEIGHTS[fold]))

# ソート
all_preds.sort(key=lambda x: x[0])
preds_list = [p[1] for p in all_preds]
used_folds = [p[0] for p in all_preds]

print(f"\n{'='*50}")
print(f"✅ Inference complete in {time.time() - total_start:.1f}s")
print(f"Used folds: {used_folds}")

## Ensemble & Physics Constraints

In [ ]:
# アンサンブル
print("\n🔄 Creating ensemble...")
total_weight = sum([CFG.FOLD_WEIGHTS[f] for f in used_folds])
ensemble = np.sum(preds_list, axis=0) / total_weight

# パス取得
paths = test_wide["image_path"].tolist()

# DataFrame作成
preds_wide = pd.DataFrame(ensemble, columns=CFG.TARGETS)
preds_wide.insert(0, 'image_path', paths)

# 物理制約（訓練と統一）
print("🔬 Applying physics constraints...")
green = preds_wide['Dry_Green_g'].values
dead = preds_wide['Dry_Dead_g'].values
clover = preds_wide['Dry_Clover_g'].values

# 制約適用
gdm_calc = green + clover
total_calc = green + dead + clover

# 予測値との混合（訓練と同じ重み）
preds_wide['GDM_g'] = 0.8 * preds_wide['GDM_g'] + 0.2 * gdm_calc
preds_wide['Dry_Total_g'] = 0.8 * preds_wide['Dry_Total_g'] + 0.2 * total_calc

# 非負制約
for col in CFG.TARGETS:
    preds_wide[col] = preds_wide[col].clip(lower=0)

# Long format変換
preds_long = preds_wide.melt(
    id_vars=['image_path'],
    value_vars=CFG.TARGETS,
    var_name='target_name',
    value_name='target'
)

## Create Submission

In [ ]:
# マージ
submission = pd.merge(
    test_df[['sample_id', 'image_path', 'target_name']],
    preds_long,
    on=['image_path', 'target_name'],
    how='left'
)

# 最終処理
submission = submission[['sample_id', 'target']]
submission['target'] = submission['target'].fillna(0.0).clip(lower=0)
submission = submission.sort_values('sample_id').reset_index(drop=True)

# 保存
submission.to_csv("submission.csv", index=False)

print(f"\n✅ Saved: submission.csv")
print(f"Shape: {submission.shape}")
print(f"\nFirst 10 rows:")
print(submission.head(10))

# 統計情報
print(f"\n📊 Stats:")
stats = submission['target'].describe()
print(f"Mean: {stats['mean']:.3f}")
print(f"Std:  {stats['std']:.3f}")
print(f"Min:  {stats['min']:.3f}")
print(f"Max:  {stats['max']:.3f}")

print(f"\n⏱️ Total time: {time.time() - total_start:.1f}s")

## Summary

### 🔧 修正内容

#### 1. KaggleCompatibleMambaBlock
- 外部依存なしのGRUベース実装
- Kaggle環境で動作保証

#### 2. 統一ディメンション
- `POOL_OUTPUT_DIM = 1920`で訓練と推論を統一
- モデル互換性の確保

#### 3. 物理制約
- 訓練と同じ重み（0.8/0.2）で適用
- GDM = Green + Clover
- Total = Green + Dead + Clover

#### 4. 安全なモデルロード
- 形状チェック付き
- エラーハンドリング強化
- 互換性の詳細ログ出力